# 한 대화에서 반품 신청 이어 가기

새한님, **`USE = "fake"`로 위에서부터 실행하고 Solar 셀은 건너뛰세요.** 배송된 주문을 조회한 뒤 “그거 반품할게요”라고 말해 봅니다. 첫 신청은 마음을 바꿔 거절합니다. 다른 주문의 반품 조건도 확인하고, 처음 주문을 다시 신청해 이번에는 승인합니다.

[대화 기억과 사람의 승인 실습](hitl_memory_playground.ipynb)에서 본 메시지, 상태, 체크포인터, `thread_id`, 도구 호출, 중단과 재개를 그대로 사용합니다. 이번에 새로 나오는 용어만 아래에 적었습니다.

| 용어 | 여기서 뜻하는 것 |
|---|---|
| 반품 자격 | 배송일, 상품 분류, 기존 접수 여부를 확인해서 내리는 판정입니다. |
| 기준일 | 배송 후 며칠 지났는지 계산하는 날짜입니다. 같은 결과를 보도록 고정합니다. |
| 반품 접수 | 주문 상태를 `반품접수`로 바꾸는 작업입니다. 이 실습에는 접수를 되돌리는 도구가 없습니다. |

한 사람이 상담을 이어 가는 상황입니다. 주문은 노트북 안의 예시 자료이고, 날짜와 상품 분류도 실습용입니다. 커널을 다시 시작하면 대화와 주문 변경을 처음부터 실습할 수 있습니다.

## 준비

작업 폴더를 shopagent로 맞춥니다. 설치된 파이썬과 패키지 버전이 출력됩니다. `show()`는 메시지마다 사람·모델·도구의 말을 한 줄씩 보여 줍니다. 모델의 `tool_calls`에는 요청한 도구 이름, 입력값, 호출 ID가 들어 있습니다. `ToolMessage`의 `name`과 `status`는 어느 도구가 어떤 형태로 응답했는지 보여 줍니다. 반품이 실제로 접수됐는지는 응답 내용과 주문 상태를 함께 보세요.

`thread_config()`는 매번 새 대화 이름표를 만듭니다. 대화를 이어 갈 때는 한 번 만든 설정을 계속 넘깁니다.

In [ ]:
import json
import os
import re
import subprocess
import sys
from datetime import date
from importlib.metadata import version
from pathlib import Path
from threading import Lock
from typing import Annotated
from uuid import uuid4

sys.dont_write_bytecode = True

REPO = Path("/home/home/projects/shopagent")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

print("Python:", sys.version.split()[0])
for package in ("langchain", "langchain-core", "langgraph", "langchain-openai", "ipykernel"):
    print(f"{package}: {version(package)}")


def show(messages):
    for message in messages:
        line = f"{message.type}: {message.content}"
        if isinstance(message, AIMessage) and message.tool_calls:
            line += " | tool_calls=" + json.dumps(message.tool_calls, ensure_ascii=False)
        if isinstance(message, ToolMessage):
            line += f" | name={message.name} status={message.status}"
        print(line.replace("\n", " ↵ "))


def thread_config(label):
    return {"configurable": {"thread_id": f"{label}-{uuid4().hex[:8]}"}}

### Gemma

`LocalLLM(port=18380).client`로 shopagent와 같은 로컬 모델 연결 객체를 준비합니다. `/health`를 최대 2초 동안 확인합니다. 상태 코드가 나오면 서버가 응답한 것입니다. 연결에 실패하면 서버 시작 안내가 나오고 다음 셀로 넘어갈 수 있습니다.

나중에 Gemma로 실습하려면 별도 터미널에서 아래 명령으로 서버를 실행하세요. 서버를 끌 때는 그 터미널에서 **Ctrl+C**를 누르세요. 가짜 모델로 진행할 때는 서버가 필요 없습니다.

```bash
/home/home/llama.cpp/build/bin/llama-server -m /home/home/models/gemma-4-12b-it-Q4_K_M.gguf --jinja -ngl 99 -ctk q8_0 -ctv q8_0 -fa on -np 1 -c 16384 --host 127.0.0.1 --port 18380 --n-cpu-moe 0 --chat-template-kwargs '{"enable_thinking": false}' --reasoning-budget 0
```

In [ ]:
import httpx
from shop.agents import LocalLLM

gemma_llm = LocalLLM(port=18380).client
print("Gemma 연결 객체 준비 완료. 모델 호출은 하지 않았습니다.")
try:
    with httpx.Client(trust_env=False) as health_client:
        health_response = health_client.get("http://127.0.0.1:18380/health", timeout=2)
    print(f"llama-server 켜짐 (상태 코드: {health_response.status_code})")
except httpx.HTTPError:
    print("llama-server 꺼짐: 위의 별도 터미널 명령으로 서버를 실행하세요.")

### Solar

가짜 모델로 실습할 때는 이 셀을 건너뛰세요. Solar를 준비할 때는 이전 실습처럼 셸에서 `upstage.env`를 불러와 키를 환경 변수에 설정합니다. 화면에는 키가 준비됐는지 `True` 또는 `False`만 나옵니다. 키가 없으면 모델 선택을 가짜 모델로 두라는 안내가 나옵니다. 키와 환경 변수 내용은 출력하지 마세요.

연결 객체에는 shopagent의 `solar-pro4` 설정을 사용합니다. **Solar를 선택한 뒤 실습에서 모델을 호출하면 비용이 듭니다.**

In [ ]:
from shop.servers import CONFIGS

def load_solar_key():
    try:
        result = subprocess.run(
            ["bash", "-c", 'set -a; . ./upstage.env; set +a; printf %s "$UPSTAGE_API_KEY"'],
            cwd=REPO, capture_output=True, text=True, timeout=5,
        )
        if result.stdout.strip():
            os.environ["UPSTAGE_API_KEY"] = result.stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        pass
    return bool(os.environ.get("UPSTAGE_API_KEY", "").strip())

solar_loaded = load_solar_key()
print("Upstage 키 loaded:", solar_loaded)
solar_llm = LocalLLM(config=CONFIGS["solar-pro4"]).client if solar_loaded else None
if solar_llm is None:
    print("키가 없어 Solar를 준비하지 못했습니다. USE = 'fake'로 진행하세요.")

### 가짜 모델

`ReturnChatModel`은 `BaseChatModel`의 틀을 써서 정해 둔 답변과 도구 요청을 만듭니다. `bind_tools()`는 에이전트가 도구를 연결할 때 호출됩니다. 이 모델은 아래 세 도구의 이름을 규칙에 사용합니다. 준비 메시지가 나오면 바로 실습할 수 있습니다.

주문 번호를 말하면 그 번호를 씁니다. “그거 반품할게요”처럼 번호가 없으면 받은 메시지에서 가장 최근에 확인된 주문을 찾습니다. 그런 기록이 없으면 번호를 묻습니다. 자격 확인 뒤에 “반품할게요”처럼 코드에 적어 둔 신청 문구까지 찾으면 접수 도구를 요청합니다. 자격만 물었거나 반품이 불가능하면 자격 확인 결과를 설명합니다. 모델 안에는 대화를 보관하는 변수가 없습니다.

In [ ]:
class ReturnChatModel(BaseChatModel):
    @property
    def _llm_type(self):
        return "return-scripted"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self

    def _tool_call(self, name, order_id, message_count):
        return AIMessage(content="", tool_calls=[{
            "name": name,
            "args": {"order_id": order_id},
            "id": f"call-{message_count}-{name}",
            "type": "tool_call",
        }])

    def _recent_order(self, messages):
        for message in reversed(messages):
            if not isinstance(message, ToolMessage) or message.status == "error":
                continue
            if message.name in ("get_order", "check_return_eligibility"):
                result = json.loads(message.content)
                if result["found"]:
                    return result["order_id"]
        return None

    def _reply(self, messages):
        latest = messages[-1]
        if isinstance(latest, ToolMessage):
            if latest.status == "error":
                return AIMessage(content="요청하신 대로 반품을 신청하지 않았어요.")

            result = json.loads(latest.content)
            if latest.name == "check_return_eligibility" and result["eligible"]:
                customer_text = next(str(message.content) for message in reversed(messages)
                                     if isinstance(message, HumanMessage))
                request_phrases = ("반품할게요", "반품하고 싶어요", "반품해 주세요", "반품 신청해 주세요")
                if any(phrase in customer_text for phrase in request_phrases):
                    return self._tool_call("request_return", result["order_id"], len(messages))
            return AIMessage(content=result["message"])

        text = str(latest.content)
        match = re.search(r"(?<![A-Za-z0-9])O[0-9]+(?![A-Za-z0-9])", text)
        order_id = match.group() if match else None
        if "반품" in text:
            order_id = order_id or self._recent_order(messages)
            if order_id:
                return self._tool_call("check_return_eligibility", order_id, len(messages))
        elif order_id:
            return self._tool_call("get_order", order_id, len(messages))
        return AIMessage(content="어느 주문인가요? 주문 번호를 알려 주세요.")

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = self._reply(messages)
        return ChatResult(generations=[ChatGeneration(message=reply)])


fake_llm = ReturnChatModel()
print("가짜 모델을 준비했습니다.")

### 이번 실습에서 쓸 모델

`USE`에는 `fake`, `gemma`, `solar` 중 하나를 넣습니다. 출력에서 어떤 모델을 골랐는지 확인하세요. 준비되지 않은 모델을 고르면 선택 셀에 안내가 나옵니다. 선택을 바꾸면 이 셀부터 아래로 다시 실행하세요. 이어지는 메시지 수와 답변 예시는 가짜 모델 기준입니다. 실제 모델은 다른 답변이나 도구 호출을 낼 수 있습니다.

In [ ]:
USE = "fake"  # "fake" | "gemma" | "solar"
models = {"fake": fake_llm, "gemma": globals().get("gemma_llm"), "solar": globals().get("solar_llm")}
assert USE in models, "USE는 fake, gemma, solar 중 하나여야 합니다."
llm = models[USE]
assert llm is not None, "선택한 모델의 준비 셀을 실행하거나 USE = 'fake'로 바꾸세요."
print("선택한 모델:", USE)

## 반품 조건과 도구 세 개

기준일은 **2026년 11월 25일**입니다. 반품하려면 배송완료 상태여야 하고, 배송 당일을 포함해 **0~7일째**여야 합니다. 위생용품은 제외하고, 이미 반품 접수된 주문도 다시 접수하지 않습니다.

- **O001 머그컵**은 배송 3일째라 신청할 수 있습니다.
- **O002 면 티셔츠**는 배송 14일째라 기간이 지났습니다.
- **O003 칫솔**은 배송 2일째지만 제외 상품입니다.

`get_order`와 `check_return_eligibility`는 주문을 읽습니다. `request_return`은 실행할 때 자격을 다시 확인하고, 가능하면 상태를 `반품접수`로 바꿉니다. 확인을 기다리는 동안 주문 상태가 바뀔 수 있기 때문입니다. 접수 함수의 재검사와 상태 변경은 잠금으로 묶어 동시에 두 번 접수되지 않게 합니다. `tool_runs`에는 실제로 실행에 들어간 도구와 주문 번호가 순서대로 남습니다. 주문 딕셔너리는 같은 커널의 모든 대화에서 공유됩니다.

In [ ]:
REFERENCE_DATE = date(2026, 11, 25)
RETURN_WINDOW_DAYS = 7
EXCLUDED_CATEGORIES = {"위생용품"}

orders = {
    "O001": {"product": "머그컵", "category": "생활용품",
             "delivered_on": "2026-11-22", "status": "배송완료"},
    "O002": {"product": "면 티셔츠", "category": "의류",
             "delivered_on": "2026-11-11", "status": "배송완료"},
    "O003": {"product": "칫솔", "category": "위생용품",
             "delivered_on": "2026-11-23", "status": "배송완료"},
}
tool_runs = []
return_lock = Lock()
OrderId = Annotated[str, "조회하거나 반품할 주문 번호. 예: O001"]


def return_blocker(order):
    if order is None:
        return "주문을 찾을 수 없어요."
    if order["status"] == "반품접수":
        return "이미 반품이 접수됐어요."
    if order["status"] != "배송완료" or not order["delivered_on"]:
        return "배송이 완료된 뒤 반품을 신청할 수 있어요."

    days = (REFERENCE_DATE - date.fromisoformat(order["delivered_on"])).days
    if days < 0:
        return "기준일 이후의 배송 건이에요. 배송일을 확인해 주세요."
    if days > RETURN_WINDOW_DAYS:
        return f"배송 후 {days}일이 지나 반품 기간({RETURN_WINDOW_DAYS}일)을 넘겼어요."
    if order["category"] in EXCLUDED_CATEGORIES:
        return f"{order['category']}은 반품 대상에서 제외돼요."
    return None


@tool
def get_order(order_id: OrderId) -> dict:
    """주문 번호로 상품명, 배송일, 상태를 조회합니다."""
    tool_runs.append(("get_order", order_id))
    order = orders.get(order_id)
    if order is None:
        return {"order_id": order_id, "found": False, "message": f"{order_id} 주문을 찾을 수 없어요."}

    message = f"{order_id} {order['product']} 주문은 {order['status']} 상태예요."
    if order["delivered_on"]:
        message += f" 배송일은 {order['delivered_on']}이에요."
    return {"order_id": order_id, "found": True, **order, "message": message}


@tool
def check_return_eligibility(order_id: OrderId) -> dict:
    """배송일, 제외 상품, 기존 반품 접수 여부를 확인합니다."""
    tool_runs.append(("check_return_eligibility", order_id))
    order = orders.get(order_id)
    blocker = return_blocker(order)
    message = f"{order_id} 주문의 반품을 신청할 수 있어요."
    if blocker:
        message = f"{order_id} 주문은 반품할 수 없어요. {blocker}"
    return {"order_id": order_id, "found": order is not None,
            "eligible": blocker is None, "message": message}


@tool
def request_return(order_id: OrderId) -> dict:
    """고객 확인 후 주문을 반품 접수합니다. 접수 전에 자격을 다시 확인합니다."""
    with return_lock:
        tool_runs.append(("request_return", order_id))
        order = orders.get(order_id)
        blocker = return_blocker(order)
        if blocker:
            return {"order_id": order_id, "ok": False,
                    "message": f"{order_id} 반품을 접수하지 않았어요. {blocker}"}

        order["status"] = "반품접수"
        return {"order_id": order_id, "ok": True, "status": order["status"],
                "message": f"{order_id} 반품 신청을 접수했어요."}

### 접수 직전에 멈추는 에이전트

`create_agent`로 모델과 도구를 연결합니다. `HumanInTheLoopMiddleware`는 `request_return` 한 건마다 멈춥니다. 조회와 자격 확인은 바로 실행됩니다. 확인 카드에는 주문 번호와 상품명이 표시됩니다. 카드 함수는 재개할 때 다시 호출될 수 있어 주문을 읽기만 합니다.

`InMemorySaver`는 대화와 대기 중인 중단을 커널 메모리에 저장합니다. 아래 셀을 다시 실행하면 새 저장소를 씁니다. 출력은 에이전트가 준비됐다는 안내입니다.

In [ ]:
return_prompt = (
    "고객의 주문을 조회하고 반품 신청을 돕는 상담원입니다. 한국어로 짧게 답하세요. "
    "주문 조회에는 get_order를 사용하세요. 고객이 주문 번호를 말하면 그 번호를 쓰세요. "
    "'그거'는 앞 대화의 주문이 분명할 때만 연결하고 모호하면 주문 번호를 물으세요. "
    "반품 가능 여부를 묻거나 반품을 신청할 때는 매번 check_return_eligibility를 먼저 호출하세요. "
    "자격이 없으면 도구가 알려 준 이유를 설명하세요. "
    "신청 의사가 있고 자격이 있으면 request_return을 한 번 호출하세요. 확인은 프로그램이 받습니다. "
    "고객이 거절하면 실행되지 않았다고 답하고 새 신청이 올 때까지 다시 호출하지 마세요. "
    "접수 도구 결과의 ok가 true일 때만 반품이 접수됐다고 답하세요."
)


def describe_return(call, state, runtime):
    order_id = call["args"]["order_id"]
    product = orders.get(order_id, {}).get("product", "상품 미확인")
    return f"{order_id} {product} 반품을 접수할까요? 접수 후에는 이 실습에서 되돌릴 수 없어요."


return_agent = create_agent(
    llm,
    [get_order, check_return_eligibility, request_return],
    system_prompt=return_prompt,
    middleware=[HumanInTheLoopMiddleware(
        interrupt_mode="per_call",
        interrupt_on={"request_return": {
            "allowed_decisions": ["approve", "reject"],
            "description": describe_return,
        }},
    )],
    checkpointer=InMemorySaver(),
)
print("반품 실습 에이전트를 준비했습니다.")

### 매 단계에서 무엇을 볼까

`get_state(config)`로 그 대화의 최신 상태를 읽습니다. 아래 도우미는 이번에 추가된 메시지를 보여 준 뒤, 저장된 **전체 메시지 수**와 `next`(다음 노드), `interrupts`(아직 답이 없는 중단 수)를 출력합니다. 대기 중이면 중단 ID와 확인 카드도 보입니다. 마지막 줄에는 이번 단계에서 실제로 실행된 도구가 나옵니다. 최신 데모 결과는 다음 셀부터 확인합니다.

`send_turn()`은 새 고객 메시지를 추가합니다. 이미 중단이 있으면 먼저 결정을 전달하라는 안내가 나옵니다. `resume_return()`은 중단이 정확히 한 건일 때 같은 설정에 `Command(resume={중단ID: 결정})`을 보냅니다. 0건이나 여러 건이면 중단 수를 알려 줍니다. 결정은 중단에 대한 응답이라 새 `HumanMessage`로 추가되지 않습니다. 거절 이유는 도구 응답에 남습니다.

In [ ]:
def show_progress(config, previous, tool_offset):
    snapshot = return_agent.get_state(config)
    messages = snapshot.values.get("messages", [])
    show(messages[len(previous.values.get("messages", [])):])
    print("전체 메시지 수:", len(messages), "| next:", snapshot.next,
          "| interrupts:", len(snapshot.interrupts))
    for pending in snapshot.interrupts:
        print(f"확인 카드 [{pending.id}] {pending.value['description']}")
    print("이번 단계에서 실행한 도구:", tool_runs[tool_offset:])
    return snapshot


def send_turn(config, customer_text):
    previous = return_agent.get_state(config)
    if previous.interrupts:
        print("대기 중인 확인에 먼저 답해 주세요.")
        return previous

    tool_offset = len(tool_runs)
    return_agent.invoke({"messages": [HumanMessage(content=customer_text)]}, config)
    return show_progress(config, previous, tool_offset)


def resume_return(config, decision):
    previous = return_agent.get_state(config)
    if len(previous.interrupts) != 1:
        print("중단이 한 건일 때만 재개합니다. 현재 중단 수:", len(previous.interrupts))
        return previous

    pending = previous.interrupts[0]
    tool_offset = len(tool_runs)
    return_agent.invoke(Command(resume={pending.id: decision}), config)
    return show_progress(config, previous, tool_offset)

## 1. 주문이 어떻게 됐는지 묻기

O001 머그컵을 조회합니다. 첫 상태에는 고객 질문, 조회 요청, 조회 결과, 모델 안내까지 **4개 메시지**가 저장됩니다. `next`가 `()`이고 중단이 0건입니다. 실행 목록에는 `get_order` 한 건이 보입니다. 이 셀에서 만든 `return_config`를 계속 씁니다.

In [ ]:
return_config = thread_config("return")
lookup_state = send_turn(return_config, "주문 O001 배송 상태를 알려 주세요.")

## 2. “그거 반품할게요”라고 말하기

이번 말에는 주문 번호가 없습니다. 모델은 앞선 조회 결과에서 O001을 찾고 반품 자격을 확인합니다. 가능하다는 결과를 받은 뒤 접수 요청을 내므로 저장된 메시지는 **8개**가 됩니다.

대기 중인 중단 1건과 확인 카드를 보세요. `next`에는 `tools`가 남아 있습니다. 실행 목록에는 자격 확인만 있고 맨 아래 O001 상태는 아직 `배송완료`입니다. 모델이 요청을 만들었고, 미들웨어가 접수 도구 실행을 보류한 상태입니다.

In [ ]:
first_request_state = send_turn(return_config, "그거 반품할게요.")
print("O001 주문 상태:", orders["O001"]["status"])

## 3. 마음이 바뀌어서 거절하기

`reject`에 고객의 거절 이유를 넣습니다. 중단이 풀리면 기본 미들웨어가 영어 안내와 고객의 이유를 담은 도구 응답을 만듭니다. `status=error`는 여기서 고객이 거절했다는 표시입니다. 이어서 모델이 접수하지 않았다고 안내합니다.

메시지는 **10개**가 되고, 중단 0건에 이번 단계 실행 목록은 비어 있습니다. O001은 그대로 `배송완료`입니다.

In [ ]:
rejected_state = resume_return(return_config, {
    "type": "reject",
    "message": "생각이 바뀌었어요. 지금은 반품하지 않을게요.",
})
print("O001 주문 상태:", orders["O001"]["status"])

## 4. 다른 주문의 반품 조건 확인하기

같은 대화에서 이번에는 O002를 지정합니다. 자격 도구가 배송 후 14일이 지났다고 알려 주고 모델이 이유를 설명합니다. 메시지가 **14개**로 늘고 중단은 0건입니다. 실행 목록에는 `check_return_eligibility`가 보입니다. 기간을 넘긴 주문에서는 확인 카드까지 갈 일이 없습니다.

In [ ]:
ineligible_state = send_turn(return_config, "그럼 O002 주문을 반품하고 싶어요.")

## 5. 처음 주문을 다시 신청하기

O002를 방금 확인했으므로 이번에는 O001을 직접 말합니다. 거절했던 건과 별개의 새 신청입니다. 같은 대화의 메시지가 **18개**로 늘어납니다. 모델은 자격을 다시 확인하고, 첫 카드와 다른 중단 ID가 생깁니다. 이때 실행 목록에는 아직 O001의 접수 도구가 없습니다.

In [ ]:
second_request_state = send_turn(return_config, "아까 조회한 O001 머그컵을 다시 반품 신청해 주세요.")

## 6. 이번에는 승인하기

`approve`로 지금 대기 중인 접수를 허가합니다. `request_return`이 실행되고 조건이 여전히 맞으면 응답에 `ok: true`가 나옵니다. 모델도 접수됐다고 안내합니다. 저장된 메시지는 **20개**, 중단은 0건입니다.

맨 아래에서 O001이 `반품접수`로 바뀌었는지 보세요. 전체 실행 기록에서 `request_return`은 한 번입니다. 두 확인 카드 중 실제 실행으로 이어진 것은 두 번째입니다.

In [ ]:
approved_state = resume_return(return_config, {"type": "approve"})
print("O001 주문 상태:", orders["O001"]["status"])
print("지금까지 실행한 도구:", tool_runs)

## 7. 새 대화에서 같은 말을 해 보기

같은 에이전트에서 `thread_config()`를 다시 부릅니다. 새 이름표로 읽은 첫 상태에는 메시지가 **0개**입니다. 2번에서와 같은 말을 보내면 이번에는 주문 번호를 물어봅니다. 이전 조회 메시지가 이 대화에 들어 있지 않기 때문입니다. 질문과 답변 **2개**만 저장되고, 중단과 도구 실행은 모두 없습니다. 주문 딕셔너리에는 앞서 승인한 접수가 그대로 남아 있습니다.

In [ ]:
new_config = thread_config("new-return")
empty_state = return_agent.get_state(new_config)
print("말하기 전 저장된 메시지 수:", len(empty_state.values.get("messages", [])))
new_state = send_turn(new_config, "그거 반품할게요.")

### shopagent에서 이어서 볼 코드

| 실습에서 본 것 | 프로젝트에서 찾을 곳 |
|---|---|
| 로컬 모델 연결과 Solar 설정 | [shop/agents.py](../shop/agents.py)의 `LocalLLM`, [shop/servers.py](../shop/servers.py)의 `CONFIGS` |
| 조회, 자격 확인, 접수 시 재검사 | [shop/tools.py](../shop/tools.py)의 `Shop.get_order`, `Shop.check_refund_eligibility`, `Shop.request_refund`. 프로젝트에서는 접수 도구를 `request_refund`라고 부릅니다. |
| 접수 전에 확인 카드 표시 | [shop/graph_v2.py](../shop/graph_v2.py)의 `build_agent`. [shop/tools.py](../shop/tools.py)의 `IRREVERSIBLE`에 들어 있는 도구가 확인 대상입니다. |
| 대화와 중단을 저장하고 이어 가기 | [shop/graph_v2.py](../shop/graph_v2.py)의 `ScenarioGraph`, `ScenarioGraph.run_turn`, `ScenarioGraph.resume_turn` |

## 직접 바꿔 볼 것

실험은 `USE = "fake"`로 하나씩 하세요. 실험 사이에는 바꾼 내용을 원래대로 돌리고, **`return-tools`부터** 다시 실행해 주문과 저장소를 새로 만드세요.

1. **같은 말, 다른 대화** — `new-thread` 셀의 `new_config = thread_config("new-return")`를 `new_config = return_config`로 바꾸세요. 시작 메시지 수가 0에서 20으로 바뀝니다. 모델은 가장 최근에 확인한 O001을 찾고, 이미 접수됐다고 답합니다. 접수 도구는 실행되지 않습니다.
2. **상품 분류 때문에 반품이 안 되는 경우** — `ineligible` 셀의 O002를 O003으로 바꾸세요. 배송한 지 이틀이어도 위생용품이라는 이유로 반품할 수 없습니다. 자격 확인 한 건, 중단 0건을 확인하세요.
3. **7일째와 8일째** — `return-tools` 셀에서 O002의 `delivered_on`을 `"2026-11-18"`로 바꾸고 `ineligible` 셀까지만 실행하세요. 배송 후 7일째라 확인 카드가 뜹니다. 다시 날짜를 `"2026-11-17"`로 바꾸고 같은 구간을 실행하면 8일째라 카드 없이 기간 초과 이유가 나옵니다.
4. **첫 신청에서 바로 승인** — `reject` 셀의 결정 딕셔너리를 `{"type": "approve"}`로 바꾸세요. 이 셀에서 O001이 접수됩니다. `rerequest` 셀에서는 이미 접수됐다는 답변이 나오고 새 중단은 0건입니다.
5. **두 번 다 거절** — `approve` 셀의 결정을 `{"type": "reject", "message": "이번에도 접수하지 않을게요."}`로 바꾸세요. 두 번 카드가 떴어도 전체 실행 기록에 접수 도구가 한 번도 나오지 않습니다. O001은 `배송완료`로 남습니다.
6. **승인 대기 중 상품 분류 정정** — `approve` 셀의 첫 줄에 `orders["O001"]["category"] = "위생용품"`을 넣으세요. 승인하면 접수 도구가 실행되고, 다시 검사한 뒤 `ok: false`를 돌려줍니다. 도구 메시지의 `status=success`는 결과가 잘 전달됐다는 표시입니다. 주문 상태는 `배송완료`로 남습니다.
7. **자격만 묻기** — `pause-return` 셀의 고객 말을 “그거 반품 가능할까요?”로 바꾸고 그 셀까지만 실행하세요. 모델이 앞 대화에서 O001을 찾고 반품 가능하다고 답합니다. 실행 목록에는 자격 도구 한 건이 있고 중단은 0건입니다.